# Tessera JAX backend — Tier 2: jit-compiled trees

**Status (2026-05-24):** Tier 2 has landed.

**Why Tier 2 exists:** if you ran `tessera_jax_tier1.ipynb` and noticed JAX was SLOWER than numpy, that's expected. Eager JAX has ~10-100 μs of Python-side dispatch overhead PER OP, so a 20-op tree pays 200μs-2ms of overhead before any compute. Numpy pays <1 μs per op. For SR-scale trees on SR-scale data (N≈60K), numpy wins eager-JAX every time.

**Tier 2's fix:** `jax.jit` the entire tree into one XLA kernel. After this, the 20 separate op dispatches collapse to a single kernel launch. Per-call overhead drops from ms to μs.

**What this notebook shows:**
- Tier 1 (eager) vs numpy: numpy usually wins
- Tier 2 (jit) vs numpy: jit wins by 10-100× depending on tree size and GPU
- The crossover point: when do you need jit vs raw numpy?

**Honest expectation on Colab T4:**
- Eager JAX `evaluate(tree, env_jax)`: SLOWER than numpy by 2-10× (this is correct, not a bug)
- jit JAX `evaluate_jit(tree, env_jax)`: faster than numpy by 5-50× after warmup
- Compile time (cold path): ~100-500ms per unique tree topology; cached after

## Setup

In [ ]:
!nvidia-smi || echo 'No GPU detected — switch runtime to GPU'

In [ ]:
# --force-reinstall --no-deps ensures Colab picks up the latest tessera
# even when the version string hasn't changed (avoids stale-cache traps).
!pip install --force-reinstall --no-deps -q git+https://github.com/davechendatascience/tessera.git
!pip install -q --upgrade "jax[cuda12]"

# After this cell runs: restart the Colab runtime (Runtime menu →
# Restart session) so Python's module cache picks up the fresh
# tessera. Then re-run from this cell forward.

In [ ]:
import jax, jax.numpy as jnp
import numpy as np
import time

print('JAX version:', jax.__version__)
print('Devices:', jax.devices())

## The same tree, three eval paths

`y = tanh(a + b) - sqrt(|c|) + log(d² + 1)`

In [ ]:
from tessera.expression import (
    Var, Const, BinOp, UnOp, evaluate, evaluate_jit, compile_tree,
    clear_jit_cache,
)

tree = BinOp('add',
    BinOp('sub',
        UnOp('tanh', BinOp('add', Var('a'), Var('b'))),
        UnOp('sqrt', Var('c'))),
    UnOp('log', BinOp('add', BinOp('mul', Var('d'), Var('d')), Const(1.0)))
)
print('tree =', tree)

In [ ]:
# Three sample sizes to show the crossover
import pandas as pd

results = []
for N in [1_000, 10_000, 60_000, 600_000]:
    rng = np.random.default_rng(0)
    env_np = {k: rng.standard_normal(N).astype(np.float32) for k in ['a','b','c','d']}
    env_jax = {k: jnp.asarray(v) for k, v in env_np.items()}
    
    # 1) numpy baseline
    _ = evaluate(tree, env_np)
    t0 = time.time()
    for _ in range(50):
        y_np = evaluate(tree, env_np)
    t_np = (time.time() - t0) / 50

    # 2) JAX eager (Tier 1)
    _ = evaluate(tree, env_jax).block_until_ready()
    t0 = time.time()
    for _ in range(50):
        _ = evaluate(tree, env_jax).block_until_ready()
    t_eager = (time.time() - t0) / 50

    # 3) JAX jit (Tier 2)
    clear_jit_cache()
    t_compile_start = time.time()
    _ = evaluate_jit(tree, env_jax).block_until_ready()   # warmup compiles
    t_compile = time.time() - t_compile_start
    t0 = time.time()
    for _ in range(50):
        _ = evaluate_jit(tree, env_jax).block_until_ready()
    t_jit = (time.time() - t0) / 50

    results.append(dict(
        N=N,
        numpy_ms=t_np*1000,
        eager_jax_ms=t_eager*1000,
        jit_jax_ms=t_jit*1000,
        compile_ms=t_compile*1000,
        eager_vs_numpy=t_np/t_eager,
        jit_vs_numpy=t_np/t_jit,
    ))

df = pd.DataFrame(results)
print(df.to_string(index=False))

**Expected reading:**

- `eager_vs_numpy < 1` at small N (eager JAX is slower) — this is the honest behavior
- `eager_vs_numpy ≥ 1` at large N (compute dominates dispatch)
- `jit_vs_numpy > 1` at all but the smallest N — jit wins
- `compile_ms` is one-time cost; amortizes after a few calls

If `jit_vs_numpy` is not strongly positive on Colab GPU, something's wrong — open an issue.

## Why this matters for the GP loop

A GP generation evaluates each candidate ~3-10 times (mutation + const-opt). With jit:
- First eval of each unique tree: ~100ms compile + few μs
- Subsequent evals (same tree): ~few μs

So for a population of 200 unique trees × 5 evals per generation:
- numpy: 200 × 5 × ~1ms = 1000ms = **1 second/generation**
- eager JAX: 200 × 5 × ~5ms = 5000ms = **5 seconds/generation** (worse!)
- jit JAX (cold): 200 × 100ms + 200 × 4 × 50μs = **20+ seconds (first generation)**
- jit JAX (warm cache): 200 × 5 × ~50μs = 50ms = **50 ms/generation**

Tier 2 unlocks the 20× speedup PER GP GENERATION after warmup. For a 100-generation run that's 100s vs 1.7 hours. The compile-cost amortizes across generations because the cache survives.

**Tier 3** will batch evaluation across the whole population in one jit call, cutting further to ~1ms/generation on GPU. That's the path to MNIST.

## Sanity: outputs match across all three paths

In [ ]:
N = 1000
rng = np.random.default_rng(0)
env_np = {k: rng.standard_normal(N).astype(np.float32) for k in ['a','b','c','d']}
env_jax = {k: jnp.asarray(v) for k, v in env_np.items()}

y_np = evaluate(tree, env_np)
y_eager = evaluate(tree, env_jax)
y_jit = evaluate_jit(tree, env_jax)

print(f'numpy:     first 5 = {y_np[:5]}')
print(f'eager JAX: first 5 = {np.asarray(y_eager[:5])}')
print(f'jit JAX:   first 5 = {np.asarray(y_jit[:5])}')

np.testing.assert_allclose(np.asarray(y_eager), y_np, rtol=1e-3)
np.testing.assert_allclose(np.asarray(y_jit), y_np, rtol=1e-3)
print('all three match within float32 precision ✓')

## Verdict

Tier 2 (`evaluate_jit`) is the correct entry point for GPU-accelerated SR. Tier 1's eager `evaluate` is useful for correctness validation but NOT a performance path. The hierarchy is:

| Path | When to use |
|---|---|
| `evaluate(tree, env_np)` | CPU, default. Always works. |
| `evaluate(tree, env_jax)` | Correctness check that the JAX path matches numpy. NOT a speed path. |
| `evaluate_jit(tree, env_jax)` | Real performance. Compiles once per unique tree; afterward μs-per-call. |
| Batched-pop vmap (Tier 3, next) | Real performance × population. The path to MNIST. |

**Next:** Tier 3 — make a whole GP population evaluate as a single vmapped jit call.